# Pseudo-labeling the test posts

The training sentences look very different from the test posts (adversarial AUC 0.99), so we label the
477 test posts with a strong LLM (the teacher) and use those posts as seeds for a larger synthetic
dataset. The transformers we train (the students) never see these labels directly, only the synthetic
data generated from them. The instructor approved this approach.

The teacher is Claude Opus 5.5 in chat. We had no API access, so it read the posts in the chat and
wrote the labels to `labels/*.json` following [`RULES.md`](RULES.md). The rules were written without
looking at the labels of the 26 sample posts, so those posts are a blind check of the teacher.

Steps:

1. Export the texts (no labels) into batches for the teacher.
2. Load the teacher's labels and check that every string occurs in its post.
3. Score the teacher on the 26 labeled sample posts, with and without the `uncertain` strings.
4. Turn the strings into character spans and look at the label statistics.
5. Split the test posts into two seed halves (A, B), keeping reposts together.
6. Save the pseudo-labeled posts and a Kaggle file with the teacher's own predictions.

In [1]:
import ast
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT.parent))
import tools
from evaluation.metric import competition_f1, published_normalize

OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

## 1. Export texts for the teacher

In [2]:
# Only rewrites the batch files when they are missing, so labels keep matching their batch.
if not (tools.BATCHES / "sample.jsonl").exists():
    print(tools.export_batches())
batches = {p.stem: [json.loads(l) for l in p.open(encoding="utf-8")] for p in sorted(tools.BATCHES.glob("*.jsonl"))}
pd.DataFrame(
    {"posts": {k: len(v) for k, v in batches.items()}, "characters": {k: sum(len(p["text"]) for p in v) for k, v in batches.items()}}
)

,posts,characters
sample,26,7654
test_1,109,37816
test_2,110,38572
test_3,85,38245
test_4,68,36394
test_5,105,37585


## 2. Load the labels and check them

In [3]:
labels = tools.read_labels()
posts = {p["id"]: p["text"] for batch in batches.values() for p in batch}
missing_ids = sorted(set(posts) - set(labels))
print(f"labeled {len(labels)} of {len(posts)} posts; missing: {missing_ids[:10]}")

not_found = []
for pid, lab in labels.items():
    _, miss = tools.find_spans(posts[pid], lab["locations"] + lab["uncertain"])
    not_found += [(pid, s) for s in miss]
print(f"strings not found in their post: {len(not_found)}", not_found[:10])

labeled 503 of 503 posts; missing: []
strings not found in their post: 0 []


## 3. Blind check on the 26 labeled sample posts

The official metric only accepts exact strings, and its punctuation cleaning is broken (see the README),
so we strip punctuation from both ends of every prediction, as we do before every submission.

In [4]:
sample = pd.read_csv(tools.DATA / "labeling_sample.csv")
truth = [ast.literal_eval(x) for x in sample["locations"]]

def clean(strings):
    """Strip edge punctuation; tools.strip_edges keeps a quote whose partner is inside ("метро «Лісова»")."""
    stripped = [tools.strip_edges(s) for s in strings]
    return [s for s in stripped if s]


adjudicated = {pid: tools.adjudicate(lab) for pid, lab in labels.items()}
variants = {
    "locations (blind)": [clean(labels[f"s{i}"]["locations"]) for i in range(len(sample))],
    "locations + uncertain (blind)": [clean(labels[f"s{i}"]["locations"] + labels[f"s{i}"]["uncertain"]) for i in range(len(sample))],
    "after consistency pass (not blind)": [clean(adjudicated[f"s{i}"]["locations"]) for i in range(len(sample))],
}
scores = pd.DataFrame({name: competition_f1(truth, pred) for name, pred in variants.items()}).T
scores

,tp,fp,fn,precision,recall,f1
locations (blind),29.0,8.0,3.0,0.783784,0.90625,0.840580
locations + uncertain (blind),31.0,20.0,1.0,0.607843,0.96875,0.746988
after consistency pass (not blind),30.0,7.0,2.0,0.810811,0.93750,0.869565


In [5]:
# Where the teacher and the sample annotators disagree (normalized like the metric).
rows = []
for i, gold in enumerate(truth):
    pred = variants["locations (blind)"][i]
    g, p = {published_normalize(x) for x in gold}, {published_normalize(x) for x in pred}
    if g != p:
        rows.append({"post": i, "missed (in gold only)": sorted(g - p), "extra (teacher only)": sorted(p - g),
                     "uncertain": labels[f"s{i}"]["uncertain"]})
pd.DataFrame(rows)

,post,missed (in gold only),extra (teacher only),uncertain
0,5,[],"[києва, україні]",[]
1,7,[тц аркадія],[],"[ТЦ Аркадія, Індустріальний міст]"
2,9,"[бучанском, фастовском]",[бучанском районах],[Фастовском]
3,11,[],[києва],[]
4,18,[],"[боярка, вишневе, караваєві дачі, тарасівка]","[Київ-Пасажирський, Васильків-Центр]"


Pick the variant with the higher F1 on the sample for the test labels. With only 26 posts (about 30
places) one or two strings move F1 by several points, so treat a small difference as a tie and keep the
simpler `locations` variant.

In [6]:
USE_UNCERTAIN = scores.loc["locations + uncertain (blind)", "f1"] > scores.loc["locations (blind)", "f1"] + 0.02
print("use uncertain strings:", USE_UNCERTAIN)

use uncertain strings: False


## 4. Consistency pass, spans and label statistics

The five test batches were labeled by separate agents, and they disagreed on a few kinds of strings:
street names written without a type word, countries left in `uncertain`, and coordinated names such as
"Луганських та Донецьких областей". `tools.adjudicate` applies one decision everywhere (listed in
RULES.md under "Adjudication"). The decisions were made after seeing the sample's gold labels, which is
why the sample score after this pass is marked "not blind"; the blind score above is the one to report.

In [7]:
test = pd.read_csv(tools.DATA / "test.csv")
records = []
for r in test.itertuples():
    lab = adjudicated[f"t{r.text_id}"]
    strings = clean(lab["locations"] + (lab["uncertain"] if USE_UNCERTAIN else []))
    spans, _ = tools.find_spans(r.text, strings)
    records.append({
        "text_id": r.text_id,
        "text": r.text,
        "locations": sorted(set(strings)),
        "loc_markers": spans,
        "uncertain": lab["uncertain"],
    })
pseudo = pd.DataFrame(records)

n_places = pseudo["locations"].str.len()
n_mentions = pseudo["loc_markers"].str.len()
street = r"^(?:вул|вулиц|просп|пр-т|пров|пл\.|площ|бульв|б-р|наб|узвіз|шосе|ул\.|улиц|переул)"
all_strings = [s for row in pseudo["locations"] for s in row]
print(f"posts with no places: {(n_places == 0).mean():.1%}")
print(f"distinct places per post: mean {n_places.mean():.2f}, median {n_places.median():.0f}, max {n_places.max()}")
print(f"mentions per post: mean {n_mentions.mean():.2f}")
print(f"share of places with a street-type word: {pd.Series(all_strings).str.lower().str.contains(street).mean():.1%}")
print(f"uncertain strings in total: {pseudo['uncertain'].str.len().sum()}")
pd.Series(Counter(all_strings)).sort_values(ascending=False).head(20)

posts with no places: 32.9%
distinct places per post: mean 1.50, median 1, max 15
mentions per post: mean 1.75
share of places with a street-type word: 10.3%
uncertain strings in total: 203


Києві                     63
Києва                     34
України                   33
Україні                   26
Київ                      16
Київщині                  16
Україну                   10
Шевченківському районі     9
Київщини                   9
Україна                    8
Дарницькому районі         5
США                        5
Дніпро                     5
Киеве                      5
Пскові                     4
рф                         4
Святошинському районі      4
Польщі                     4
Шевченківському            3
Біла Церква                3
dtype: int64

## 5. Seed halves A and B

To check later that the student generalizes instead of memorizing the seeds, we generate synthetic
data from half A only, train on it, and compare the student's agreement with the teacher on A and on B.
Reposts share most of their text, so near-duplicate posts always go to the same half.

In [8]:
groups = tools.near_duplicate_groups(pseudo["text"].tolist())
pseudo["group"] = groups
rng = np.random.default_rng(42)
unique_groups = np.array(sorted(set(groups)))
half_a = set(rng.choice(unique_groups, size=len(unique_groups) // 2, replace=False))
pseudo["seed_half"] = ["A" if g in half_a else "B" for g in groups]
print(f"{len(unique_groups)} groups for {len(pseudo)} posts")
pseudo.groupby("seed_half").agg(posts=("text_id", "size"), places=("locations", lambda s: s.str.len().sum()))

460 groups for 477 posts


,posts,places
seed_half,,
A,239,402
B,238,314


## 6. Save

In [9]:
pseudo.to_parquet(OUT / "test_pseudo.parquet", index=False)

# The teacher's labels as a Kaggle submission: its leaderboard score is the ceiling we expect the
# students to approach, and it tells us how well the teacher's conventions match the hidden labels.
submission = pd.DataFrame({"text_id": pseudo["text_id"], "locations": pseudo["locations"].map(str)})
submission.to_csv(OUT / "submission_teacher.csv", index=False)
print("saved", OUT / "test_pseudo.parquet", "and", OUT / "submission_teacher.csv")

saved /Users/allakovalcuk/AI301/deep-learning-assignment4/pseudo_label/output/test_pseudo.parquet and /Users/allakovalcuk/AI301/deep-learning-assignment4/pseudo_label/output/submission_teacher.csv
